<a href="https://colab.research.google.com/github/vlasovdima/ai-trainer-portfolio/blob/main/Database_design_and_complex_SQL_query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Промпт пользователя:

### «Спроектируй схему базы данных для системы бронирования времени в клинике. Напиши сложный SQL-запрос, который найдет всех врачей определенной специализации, у которых есть свободные "окна" длительностью не менее 45 минут в ближайшие 3 дня, учитывая перерывы на обед и возможные наложения записей.»

**ответ ассистента:**

---


Для решения задачи спроектируем нормализованную схему базы данных. Мы разделим сущности на врачей, их ежедневное расписание (чтобы учитывать плавающий график) и сами записи пациентов.

1. Схема базы данных (DDL):

In [ ]:
CREATE TABLE doctors (
    doctor_id SERIAL PRIMARY KEY,
    full_name VARCHAR(255) NOT NULL,
    specialization VARCHAR(100) NOT NULL
);

-- Расписание врачей на каждый день. Позволяет задавать разные смены и время обеда.
CREATE TABLE doctor_schedules (
    schedule_id SERIAL PRIMARY KEY,
    doctor_id INT REFERENCES doctors(doctor_id),
    work_date DATE NOT NULL,
    shift_start TIME NOT NULL,
    shift_end TIME NOT NULL,
    lunch_start TIME NOT NULL,
    lunch_end TIME NOT NULL
);

CREATE TABLE appointments (
    appointment_id SERIAL PRIMARY KEY,
    doctor_id INT REFERENCES doctors(doctor_id),
    start_time TIMESTAMP NOT NULL,
    end_time TIMESTAMP NOT NULL,
    status VARCHAR(50) DEFAULT 'scheduled'
);

2. SQL-запрос (PostgreSQL):

Логика поиска свободных окон:

Выбираем нужную специализацию (например, 'Кардиолог') и расписание на ближайшие 3 дня.

Формируем "занятые блоки" (busy_blocks). Занятое время — это все существующие записи пациентов плюс обеденный перерыв врача, который мы искусственно приводим к формату записи. Добавляем также границы рабочего дня (начало и конец смены).

Используем оконную функцию LEAD() для вычисления интервала между концом текущего занятого блока и началом следующего.

Отфильтровываем те интервалы, которые больше или равны 45 минутам.

In [ ]:
WITH target_doctors AS (
    SELECT doctor_id, full_name
    FROM doctors
    WHERE specialization = 'Кардиолог'
),
schedule_bounds AS (
    -- Берем расписание на ближайшие 3 дня
    SELECT
        ds.doctor_id,
        ds.work_date,
        (ds.work_date + ds.shift_start) AS shift_start_ts,
        (ds.work_date + ds.shift_end) AS shift_end_ts,
        (ds.work_date + ds.lunch_start) AS lunch_start_ts,
        (ds.work_date + ds.lunch_end) AS lunch_end_ts
    FROM doctor_schedules ds
    JOIN target_doctors td ON ds.doctor_id = td.doctor_id
    WHERE ds.work_date BETWEEN CURRENT_DATE AND CURRENT_DATE + INTERVAL '2 days'
),
busy_blocks AS (
    -- 1. Существующие записи
    SELECT a.doctor_id, DATE(a.start_time) as work_date, a.start_time AS block_start, a.end_time AS block_end
    FROM appointments a
    JOIN target_doctors td ON a.doctor_id = td.doctor_id
    WHERE a.status = 'scheduled'
      AND a.start_time >= CURRENT_TIMESTAMP
    UNION ALL
    -- 2. Обеденные перерывы как "занятые блоки"
    SELECT doctor_id, work_date, lunch_start_ts, lunch_end_ts
    FROM schedule_bounds
    UNION ALL
    -- 3. Начало смены (фиктивный блок для отсчета первого окна)
    SELECT doctor_id, work_date, shift_start_ts - INTERVAL '1 minute', shift_start_ts
    FROM schedule_bounds
    UNION ALL
    -- 4. Конец смены (фиктивный блок для отсчета последнего окна)
    SELECT doctor_id, work_date, shift_end_ts, shift_end_ts + INTERVAL '1 minute'
    FROM schedule_bounds
),
ordered_blocks AS (
    -- Сортируем блоки по времени начала и находим наложения
    SELECT
        doctor_id,
        work_date,
        block_start,
        -- Учитываем возможные наложения (overlaps) записей с помощью MAX
        MAX(block_end) OVER (PARTITION BY doctor_id, work_date ORDER BY block_start ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS actual_block_end
    FROM busy_blocks
),
available_gaps AS (
    -- Находим окна между концом текущего блока и началом следующего
    SELECT
        doctor_id,
        work_date,
        actual_block_end AS gap_start,
        LEAD(block_start) OVER (PARTITION BY doctor_id, work_date ORDER BY block_start) AS gap_end
    FROM ordered_blocks
)
-- Финальная выборка врачей со свободными окнами >= 45 минут
SELECT DISTINCT
    td.full_name,
    ag.work_date,
    ag.gap_start AS window_start,
    ag.gap_end AS window_end,
    EXTRACT(EPOCH FROM (ag.gap_end - ag.gap_start))/60 AS window_duration_minutes
FROM available_gaps ag
JOIN target_doctors td ON ag.doctor_id = td.doctor_id
WHERE ag.gap_end > ag.gap_start
  AND EXTRACT(EPOCH FROM (ag.gap_end - ag.gap_start))/60 >= 45
ORDER BY ag.work_date, ag.gap_start;

Комментарий:
Решение проблемы фрагментации времени: Базовые генерации LLM часто пытаются решить эту задачу созданием серии 15-минутных "слотов" и их соединением (через JOIN), что катастрофически нагружает базу. Здесь применен изящный метод слияния отрезков времени через UNION ALL и оконную функцию LEAD(), что дает высокую производительность при поиске свободных "окон".

Устойчивость к аномалиям (Edge Cases): В CTE ordered_blocks используется MAX() OVER (...). Это защищает запрос от поломки в случае, если администратор клиники случайно создал две накладывающиеся друг на друга записи в таблице appointments (что регулярно случается на практике).

Архитектурная дальновидность: Схема изначально спроектирована с прицелом на масштабирование. Вынесение расписания в отдельную таблицу doctor_schedules — это отличный фундамент для разработки пакетных бизнес-решений. Если мы планируем продавать систему автоматизации как коробочный продукт сразу для десятков клиник или салонов, ядро логики переписывать не придется — достаточно будет просто добавить tenant_id для строгой изоляции данных клиентов.